# EU Regulatory Compliance Search — Qdrant Hybrid Search Demo

End-to-end walkthrough: chunk EUR-Lex data → generate dense + SPLADE sparse embeddings → create a Qdrant multi-vector collection → run RRF-fused hybrid queries → benchmark against Dense-only and Dense+BM25.

Run this notebook from the repository root (`qdrant/`), or in Colab after `!git clone` and `%cd qdrant`.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os

# For this demo we run an in-memory Qdrant instance (no server needed).
# For production, set QDRANT_URL / QDRANT_API_KEY to your Qdrant Cloud instance instead.
os.environ.setdefault("QDRANT_URL", ":memory:")

## 1. Prepare + chunk a small sample of EUR-Lex

In [ ]:
from src.data_prep import build_chunks
import json, pathlib

chunks = build_chunks(max_docs=25, local_dir=None, chunk_size=800, overlap=150)
pathlib.Path("data").mkdir(exist_ok=True)
with open("data/chunks.jsonl", "w") as f:
    for c in chunks:
        f.write(json.dumps(c) + "\n")
print(f"Prepared {len(chunks)} chunks")
chunks[0]

## 2. Create the Qdrant multi-vector collection (dense + sparse)

In [ ]:
from qdrant_client import QdrantClient
from src.qdrant_setup import create_collection
from src.config import load_settings

settings = load_settings()
client = QdrantClient(":memory:")
create_collection(client, settings.collection_name, recreate=True)

## 3. Embed (dense + SPLADE + BM25) and ingest

In [ ]:
import src.qdrant_setup as qs
qs.get_client = lambda: client  # reuse the in-memory client from this notebook session

from src.ingest import load_chunks, ingest

chunks = load_chunks(pathlib.Path("data/chunks.jsonl"))
ingest(chunks, batch_size=32)

## 4. Run a hybrid query

In [ ]:
import src.search as s
s.get_client = lambda: client

results = s.hybrid_search("obligations for providers of high-risk AI systems", mode="hybrid", limit=5)
for r in results:
    print(f"{r.score:.4f}  {r.doc_id}  {r.text[:120]}...")

## 5. Benchmark Dense-only vs Dense+BM25 vs Dense+SPLADE

Point `--queries` at a labeled set (see `data/eval_queries.jsonl` for the expected schema).

In [ ]:
import src.evaluate as ev
ev.hybrid_search = s.hybrid_search  # reuse the in-memory-backed search function

queries = ev.load_eval_queries(pathlib.Path("data/eval_queries.jsonl"))
report = ev.evaluate(queries, k=5)
ev.print_report(report, k=5)